# 📊 Análise Exploratória — Olist E-Commerce Brasil
> Dataset público com pedidos reais de e-commerce brasileiro  
> Período: outubro de 2016 a agosto de 2018 | Fonte: Kaggle

## 0.0. Instalando e Importando Bibliotecas Necessárias

In [34]:
import plotly.io as pio
import plotly.express as px
import plotly.graph_objects as go
import pandas as pd
import numpy as np



### 0.1. Configuração

In [35]:
# Configuração segura para garantir que os gráficos apareçam mesmo sem kaleido

pio.renderers.default = 'colab'


layout_transparente = dict(
    paper_bgcolor='rgba(0,0,0,0)',
    plot_bgcolor='rgba(0,0,0,0)',
    font=dict(color='white')
)

### 0.2. Carregando os dados

In [36]:


df = pd.read_parquet('/content/olist_receita.parquet')

# dataset COMPLETO (inclui cancelamentos)
df_diag = pd.read_parquet('/content/olist_completo.parquet')

print(f"✅ Dados carregados!")


✅ Dados carregados!


### 0.2. Resumo dos Dados Carregados

In [37]:
print(f"📊 Shape: {df.shape}")
print(f"📅 Período: {df['ano_mes'].min()} até {df['ano_mes'].max()}")
print(f"🏪 Total de pedidos únicos: {df['order_id'].nunique():,}")
print(f"💰 Receita total: R$ {df['price'].sum():,.2f}")

📊 Shape: (112086, 32)
📅 Período: 2016-10 até 2018-08
🏪 Total de pedidos únicos: 98,187
💰 Receita total: R$ 13,492,787.80


## 1.0. Análise Exploratória dos Dados

### 1.1. Visão Geral do Negócio

Métricas principais do período — receita total, ticket médio,  
avaliação média e cobertura geográfica.

In [38]:
# métricas gerais
total_pedidos = df['order_id'].nunique()
receita_total = df['price'].sum()
ticket_medio = receita_total / total_pedidos
frete_total = df['freight_value'].sum()
frete_medio = frete_total / total_pedidos
ticket_com_frete = ticket_medio + frete_medio
total_estados = df['customer_state'].nunique()
total_categorias = df['product_category'].nunique()
media_avaliacao = df[(df['pedido_item_seq'] == 1) & (df['tem_avaliacao'] == True)]['review_score'].mean()

print("=" * 50)
print("        VISÃO GERAL DO NEGÓCIO — OLIST")
print("=" * 50)
print(f"📦 Total de pedidos:        {total_pedidos:>10,}")
print(f"💰 Receita total:           R$ {receita_total:>10,.2f}")
print(f"🎯 Ticket médio:            R$ {ticket_medio:>10,.2f}")
print(f"🚚 Frete médio:             R$ {frete_medio:>10,.2f}")
print(f"🛒 Ticket médio c/ frete:   R$ {ticket_com_frete:>10,.2f}")
print(f"⭐ Avaliação média:         {media_avaliacao:>10.2f} / 5.0")
print(f"🗺️  Estados atendidos:       {total_estados:>10}")
print(f"📂 Categorias de produtos:  {total_categorias:>10}")
print("=" * 50)

        VISÃO GERAL DO NEGÓCIO — OLIST
📦 Total de pedidos:            98,187
💰 Receita total:           R$ 13,492,787.80
🎯 Ticket médio:            R$     137.42
🚚 Frete médio:             R$      22.82
🛒 Ticket médio c/ frete:   R$     160.24
⭐ Avaliação média:               4.12 / 5.0
🗺️  Estados atendidos:               27
📂 Categorias de produtos:          72


### 1.2. Distribuição de status dos pedidos

98% dos pedidos foram entregues com sucesso — indicador  
de alta eficiência operacional da plataforma.

In [39]:
import plotly.graph_objects as go

# 1. Preparação dos dados com cálculos de precisão
status_counts = df[df['pedido_item_seq'] == 1]['order_status'].value_counts().reset_index()
status_counts.columns = ['status', 'quantidade']

traducao_status = {
    'delivered': 'Entregue',
    'shipped': 'Enviado',
    'canceled': 'Cancelado',
    'unavailable': 'Indisponível',
    'invoiced': 'Faturado',
    'processing': 'Em processamento',
    'created': 'Criado',
    'approved': 'Aprovado'
}

status_counts['status_pt'] = status_counts['status'].map(traducao_status).fillna(status_counts['status'])
total_pedidos = status_counts['quantidade'].sum()
status_counts['pct'] = (status_counts['quantidade'] / total_pedidos * 100)

# Formatação da Legenda de Precisão (Tabela de Consulta)
def format_legend(row):
    pct_str = f"{row['pct']:.1f}%" if row['pct'] >= 0.1 else "<0,1%"
    return f"{row['status_pt']} ({pct_str})"

status_counts['legend_label'] = status_counts.apply(format_legend, axis=1)

# 2. Design de Cores Estratégicas (Nível Sênior)
cores_map = {
    'Entregue': '#1f77b4',         # Vibrant Blue
    'Enviado': '#808080',          # Grey
    'Faturado': '#4F4F4F',         # Graphite
    'Em processamento': '#FF8C00', # Vibrant Orange
    'Cancelado': '#363636',
    'Indisponível': '#2F2F2F',
    'Aprovado': '#F5F5F5'          # Off-White
}

cores_sequencia = [cores_map.get(s, '#808080') for s in status_counts['status_pt']]

# 3. Construção do Donut Chart High-Performance
fig = go.Figure(data=[go.Pie(
    labels=status_counts['legend_label'],
    values=status_counts['quantidade'],
    hole=0.72,
    marker=dict(colors=cores_sequencia, line=dict(color='rgba(0,0,0,0)', width=2)),
    textinfo='none',
    hoverinfo='label+value',
    sort=False,                # Mantém a ordem lógica da legenda
    showlegend=True
)])

# 4. KPI Central (Vibrante e Nítido)
fig.add_annotation(
    text="<span style='font-size:42px; font-weight:bold; color:#1f77b4;'>98,2%</span><br><span style='font-size:16px; font-weight:300; color:rgba(255,255,255,0.8);'>Entregues</span>",
    x=0.5, y=0.5, showarrow=False
)

# 5. Nota de Rodapé e Credibilidade
fig.add_annotation(
    text="Fonte: Dados Olist (Out 2016 a Ago 2018)",
    xref="paper", yref="paper",
    x=1, y=-0.1, showarrow=False,
    font=dict(size=10, color='rgba(255,255,255,0.4)'),
    align="right"
)

# 6. Layout de Consultoria (Padding e Alinhamento)
fig.update_layout(
    layout_transparente,
    title=dict(
        text="<b>Eficiência Logística: Pedidos Concluídos</b><br><span style='font-size:14px; color:rgba(255,255,255,0.6);'>Visão consolidada de status operacional</span>",
        x=0.5, y=0.95
    ),
    legend=dict(
        orientation="v",
        yanchor="middle",
        y=0.5,
        xanchor="left",
        x=1.1,
        font=dict(size=13, color='white'),
        bgcolor='rgba(0,0,0,0)'
    ),
    height=550,
    margin=dict(t=130, b=80, l=50, r=220) # Margem direita ampla para a legenda detalhada
)

fig.show()

### 1.3. Evolução de Vendas ao Longo do Tempo

Crescimento médio de 21.95% ao mês entre 2016 e 2018.  
Novembro de 2017 foi o melhor mês — impulsionado pela Black Friday.

In [40]:
# Receita mensal

receita_mensal = df.groupby('ano_mes').agg(
    receita=('price', 'sum'),
    pedidos=('order_id', 'nunique'),
    ticket_medio=('price', 'mean')
).reset_index()

# crescimento mês a mês
receita_mensal['crescimento_pct'] = receita_mensal['receita'].pct_change() * 100

print("Receita por mês:")
print(receita_mensal.to_string(index=False))

Receita por mês:
ano_mes    receita  pedidos  ticket_medio  crescimento_pct
2016-10   44507.30      290    130.138304              NaN
2017-01  120098.27      787    126.021270       169.839487
2017-02  244959.35     1718    126.528590       103.965761
2017-03  368341.32     2617    123.812208        50.368345
2017-04  353842.98     2377    133.023677        -3.936116
2017-05  502980.19     3639    122.528670        42.147850
2017-06  429916.61     3205    120.391098       -14.526135
2017-07  492287.30     3946    109.567616        14.507625
2017-08  568245.79     4272    116.396106        15.429707
2017-09  621415.91     4227    129.058341         9.356888
2017-10  660179.62     4547    124.562192         6.237965
2017-11 1003744.84     7420    116.376213        52.041173
2017-12  742183.79     5618    117.806951       -26.058520
2018-01  945456.29     7187    115.680447        27.388432
2018-02  837895.43     6624    110.292935       -11.376608
2018-03  981051.06     7168    119.7133

In [41]:
import numpy as np
import plotly.graph_objects as go

# 1. Preparação de Dados com Tradução para PT-BR
meses_pt = {
    '2016-10': 'Out 16', '2017-01': 'Jan 17', '2017-02': 'Fev 17', '2017-03': 'Mar 17',
    '2017-04': 'Abr 17', '2017-05': 'Mai 17', '2017-06': 'Jun 17', '2017-07': 'Jul 17',
    '2017-08': 'Ago 17', '2017-09': 'Set 17', '2017-10': 'Out 17', '2017-11': 'Nov 17',
    '2017-12': 'Dez 17', '2018-01': 'Jan 18', '2018-02': 'Fev 18', '2018-03': 'Mar 18',
    '2018-04': 'Abr 18', '2018-05': 'Mai 18', '2018-06': 'Jun 18', '2018-07': 'Jul 18', '2018-08': 'Ago 18'
}
eixo_x_pt = [meses_pt.get(m, m) for m in receita_mensal['ano_mes']]

# 2. Cálculos Estatísticos (Bandas de Bollinger)
window = 3
rolling_mean = receita_mensal['receita'].rolling(window=window).mean()
rolling_std = receita_mensal['receita'].rolling(window=window).std()
upper_band = rolling_mean + (rolling_std * 2)
lower_band = rolling_mean - (rolling_std * 2)

fig = go.Figure()

# Camada 1: Banda de Volatilidade (Sutil no gráfico)
fig.add_trace(go.Scatter(
    x=eixo_x_pt, y=upper_band,
    line=dict(color='rgba(255,255,255,0)'),
    showlegend=False, hoverinfo='none'
))
fig.add_trace(go.Scatter(
    x=eixo_x_pt, y=lower_band,
    fill='tonexty',
    fillcolor='rgba(128, 128, 128, 0.12)',
    line=dict(color='rgba(255,255,255,0)'),
    name='Volatilidade',
    hoverinfo='none',
    marker=dict(color='rgba(128, 128, 128, 0.4)')
))

# Camada 2: Média Móvel
fig.add_trace(go.Scatter(x=eixo_x_pt, y=rolling_mean, name='Média Móvel (3m)', line=dict(color='rgba(200, 200, 200, 0.35)', width=1.5, dash='dot')))

# Camada 3: Receita Real com Rótulo Final de Precisão
ultimo_valor = receita_mensal['receita'].iloc[-1]
ultimo_mes = eixo_x_pt[-1]
valor_formatado = f"R$ {ultimo_valor/1000:,.1f}k".replace('.', 'v').replace(',', '.').replace('v', ',')

fig.add_trace(go.Scatter(
    x=eixo_x_pt, y=receita_mensal['receita'],
    mode='lines+markers', name='Receita Real',
    line=dict(color='#1f77b4', width=3),
    marker=dict(size=8, symbol='circle', line=dict(color='white', width=1))
))

# Rótulo de dado no último ponto (Afastamento de 5px e Alinhamento Central)
fig.add_annotation(
    x=ultimo_mes, y=ultimo_valor,
    text=f"<b>{valor_formatado}</b>",
    showarrow=False,
    xanchor='left', xshift=5, yshift=0,
    font=dict(color='white', size=12)
)

# 3. Anotação Black Friday (Fina e Elegante)
bf_val = receita_mensal.loc[receita_mensal['ano_mes'] == '2017-11', 'receita'].values[0]
fig.add_annotation(
    x='Nov 17',
    y=bf_val,
    text='<b>Pico Black Friday</b><br>(~R$ 1,0M)',
    showarrow=True,
    arrowhead=2,
    arrowsize=1.0, # Seta mais discreta
    arrowwidth=1.0, # Fina e elegante
    arrowcolor='white',
    standoff=6, # Garante que a seta aponte sem tocar o marcador
    ax=0,
    ay=-65,
    bgcolor='rgba(35, 35, 35, 0.92)',
    bordercolor='rgba(255, 255, 255, 0.15)',
    borderwidth=1,
    borderpad=8,
    font=dict(color='white', size=12)
)

# 4. Nota de Rodapé e Ajuste de Margens (Respiro)
fig.add_annotation(
    text="Fonte: Dados Olist (Out 2016 a Ago 2018)",
    xref="paper", yref="paper",
    x=1, y=-0.26, showarrow=False,
    font=dict(size=10, color='rgba(255,255,255,0.4)'),
    align="right"
)

fig.update_layout(
    layout_transparente,
    title=dict(
        text='<b>Receita e Volatilidade: Impacto Decisivo da Black Friday e Crescimento Sustentado</b><br><span style="font-size:13px; color:rgba(255,255,255,0.6);">Análise mensal consolidada (Out 2016 a Ago 2018) | Média Móvel (3m) e faixa de volatilidade</span>',
        x=0.01, y=0.95
    ),
    height=550,
    margin=dict(t=120, b=130, l=80, r=95), # Margem direita e inferior ampliada para respiro
    legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='right', x=1, font=dict(size=12)),
    hovermode='closest'
)

# 5. Eixos - Eliminação de Ruído (Barra Branca/Zeroline OFF)
fig.update_xaxes(
    showgrid=False,
    tickangle=-45,
    dtick=2,
    title=None,
    tickfont=dict(color='rgba(255,255,255,0.8)'),
    showline=False, # Flutuação elegante
    zeroline=False  # Remove a barra branca vertical
)

fig.update_yaxes(
    showgrid=True, gridwidth=1, gridcolor='rgba(128, 128, 128, 0.1)',
    tickvals=[0, 200000, 400000, 600000, 800000, 1000000, 1200000],
    ticktext=['R$ 0', '200k', '400k', '600k', '800k', '1,0M', '1,2M'],
    title=None,
    tickfont=dict(color='rgba(255,255,255,0.8)'),
    showline=False,
    zeroline=False # Remove a barra branca horizontal na base
)

fig.show()

In [42]:
 # Melhor e pior mês + insights

melhor_mes = receita_mensal.loc[receita_mensal['receita'].idxmax()]
pior_mes = receita_mensal.loc[receita_mensal['receita'].idxmin()]
meses_positivos = (receita_mensal['crescimento_pct'] > 0).sum()
meses_negativos = (receita_mensal['crescimento_pct'] < 0).sum()
crescimento_medio = receita_mensal['crescimento_pct'].dropna().mean()

print("=" * 50)
print("        INSIGHTS — EVOLUÇÃO TEMPORAL")
print("=" * 50)
print(f"📈 Melhor mês:    {melhor_mes['ano_mes']} — R$ {melhor_mes['receita']:,.2f}")
print(f"📉 Pior mês:      {pior_mes['ano_mes']} — R$ {pior_mes['receita']:,.2f}")
print(f"📊 Crescimento médio mensal: {crescimento_medio:.2f}%")
print(f"✅ Meses positivos: {meses_positivos}")
print(f"❌ Meses negativos: {meses_negativos}")
print("=" * 50)

        INSIGHTS — EVOLUÇÃO TEMPORAL
📈 Melhor mês:    2017-11 — R$ 1,003,744.84
📉 Pior mês:      2016-10 — R$ 44,507.30
📊 Crescimento médio mensal: 21.95%
✅ Meses positivos: 13
❌ Meses negativos: 7


### 1.4. Receita por Categoria de Produto

Quais categorias dominam a receita, quais têm maior ticket médio  
e onde estão as oportunidades de alto valor e baixo volume.

In [43]:
import pandas as pd
import plotly.express as px

# Agrupamento e tradução
categorias = df.groupby('product_category').agg(
    receita=('price', 'sum'),
    pedidos=('order_id', 'nunique'),
    ticket_medio=('price', 'mean')
).reset_index().sort_values('receita', ascending=False)

traducao_categorias = {
    'health_beauty': 'Beleza e Saúde',
    'watches_gifts': 'Relógios e Presentes',
    'bed_bath_table': 'Cama, Mesa e Banho',
    'sports_leisure': 'Esporte e Lazer',
    'computers_accessories': 'Informática e Acessórios',
    'furniture_decor': 'Móveis e Decoração',
    'housewares': 'Utilidades Domésticas',
    'auto': 'Automotivo',
    'garden_tools': 'Ferramentas de Jardim',
    'cool_stuff': 'Brinquedos e Games'
}
categorias['product_category'] = categorias['product_category'].map(traducao_categorias).fillna(categorias['product_category'])

top10_receita = categorias.head(10).copy()

# --- Cálculos de Storytelling Executivo ---
total_top10 = top10_receita['receita'].sum()
top10_receita['pct_top10'] = (top10_receita['receita'] / total_top10 * 100).round(1)
pct_top3 = (top10_receita.head(3)['receita'].sum() / total_top10 * 100).round(0)

# Preparação do gráfico
top10_plot = top10_receita.sort_values('receita')
cores_destaque = ['#808080'] * (len(top10_plot) - 1) + ['#1f77b4']

# Formatação robusta dos rótulos (Padrão Sênior BR)
labels_custom = []
for v, p in zip(top10_plot['receita'], top10_plot['pct_top10']):
    p_str = f"{p:.1f}%".replace('.', ',')
    if v >= 1_000_000:
        valor_str = f"{v/1_000_000:.1f}M".replace('.', ',')
    else:
        valor_str = f"{v/1_000:.0f}K".replace('.', ',')
    labels_custom.append(f"R$ {valor_str} (~{p_str})")

fig = px.bar(
    top10_plot,
    x='receita',
    y='product_category',
    orientation='h',
    title=f'<b>Saúde & Beleza Lidera Faturamento Acumulado (R$ 1,3M)</b><br><span style="font-size:13px; color:rgba(255,255,255,0.6);">Top 3 categorias concentram ~{pct_top3:.0f}% das vendas líderes no período</span>',
    text=labels_custom
)

fig.update_traces(
    marker_color=cores_destaque,
    textposition='outside',
    textfont=dict(size=12, color='white'),
    cliponaxis=False,
    width=0.90 # Barras mais largas para compactação
)

fig.update_layout(
    layout_transparente,
    title_x=0,
    height=500,
    showlegend=False,
    margin=dict(l=0, r=180, t=100, b=60), # l=0 para aproximar eixo Y
    bargap=0.02 # Gap mínimo para visual sólido
)

# Remoção total do Eixo X
fig.update_xaxes(
    showgrid=False,
    showline=False,
    zeroline=False,
    showticklabels=False,
    title=None
)

fig.update_yaxes(
    ticks='outside',
    ticklen=2,
    tickcolor='rgba(0,0,0,0)',
    showgrid=False,
    showline=False,
    zeroline=False,
    title=None,
    automargin=True,
    tickfont=dict(size=12, color='white')
)

# Nota de Rodapé
fig.add_annotation(
    text="Fonte: Dados Olist (Out 2016 a Ago 2018)",
    xref="paper", yref="paper",
    x=1.15, y=-0.12, showarrow=False,
    font=dict(size=10, color='rgba(255,255,255,0.3)'),
    align="right"
)

fig.show()

In [44]:
import pandas as pd
import plotly.express as px

# 1. Preparação e Tradução de Dados
categorias_relevantes = categorias[categorias['pedidos'] >= 100].copy()
top10_ticket = categorias_relevantes.nlargest(10, 'ticket_medio').copy()

traducao_refinada = {
    'computers': 'Computadores',
    'small_appliances': 'Eletroportáteis',
    'musical_instruments': 'Instrumentos Musicais',
    'agro_industry_and_commerce': 'Agro e Indústria',
    'home_appliances_2': 'Eletrodomésticos (Linha 2)',
    'air_conditioning': 'Ar Condicionado',
    'fixed_telephony': 'Telefonia Fixa',
    'construction_tools_safety': 'Ferramentas de Construção',
    'kitchen_dining_laundry_garden_furniture': 'Móveis de Jardim/Cozinha'
}

top10_ticket['product_category'] = top10_ticket['product_category'].map(traducao_refinada).fillna(top10_ticket['product_category'])
top10_ticket_plot = top10_ticket.sort_values('ticket_medio')

# 2. Lógica de Cores e Rótulos (Padrão Sênior - Master Polish)
cores_ticket = ['#808080'] * (len(top10_ticket_plot) - 1) + ['#1f77b4']
segundo_lugar_valor = top10_ticket_plot.iloc[-2]['ticket_medio']
leader_valor = top10_ticket_plot.iloc[-1]['ticket_medio']
multiplicador = leader_valor / segundo_lugar_valor

labels_custom = []
for i, row in top10_ticket_plot.iterrows():
    valor_br = f"R$ {row['ticket_medio']:,.2f}".replace(',', 'v').replace('.', ',').replace('v', '.')
    if row['product_category'] == 'Computadores':
        # Correção Master: 2,3x (vírgula para decimal)
        mult_str = f"{multiplicador:.1f}".replace('.', ',')
        labels_custom.append(f"<b>{valor_br}</b> ({mult_str}x o 2º lugar)")
    else:
        labels_custom.append(valor_br)

# 3. Construção do Gráfico (Maximizing Data-to-Ink Ratio)
fig = px.bar(
    top10_ticket_plot,
    x='ticket_medio',
    y='product_category',
    orientation='h',
    title='<b>Computadores Lidera Ticket Médio com o Dobro do 2º Lugar</b><br><span style="font-size:13px; color:rgba(255,255,255,0.6);">Dados consolidados (Out 2016 a Ago 2018) | Mínimo de 100 pedidos</span>',
    text=labels_custom
)

fig.update_traces(
    marker_color=cores_ticket,
    textposition='outside',
    textfont=dict(size=12, color='white'),
    cliponaxis=False,
    width=0.90 # Barras mais largas para compactação
)

# 4. Layout Executivo (Aproximação e Compactação)
fig.update_layout(
    layout_transparente,
    title_x=0,
    height=500,
    showlegend=False,
    margin=dict(l=0, r=250, t=100, b=60), # l=0 para aproximar eixo Y da borda
    bargap=0.02 # Gap mínimo para visual sólido
)

fig.update_xaxes(
    showgrid=False,
    showline=False,
    zeroline=False,
    showticklabels=False,
    title=None
)

fig.update_yaxes(
    ticks='outside',
    ticklen=2,
    tickcolor='rgba(0,0,0,0)',
    showgrid=False,
    showline=False,
    zeroline=False,
    title=None,
    automargin=True,
    tickfont=dict(size=12, color='white')
)

# 5. Nota de Rodapé
fig.add_annotation(
    text="Fonte: Dados Olist (Out 2016 a Ago 2018)",
    xref="paper", yref="paper",
    x=1.25, y=-0.12, showarrow=False,
    font=dict(size=10, color='rgba(255,255,255,0.3)'),
    align="right"
)

# 6. Exportação em Alta Resolução para GitHub/Apresentação
# Descomente a linha abaixo caso queira gerar o arquivo físico
# fig.write_image("ticket_medio_senior.png", scale=3)

fig.show()

In [45]:
media_ticket = categorias_relevantes['ticket_medio'].mean()
media_pedidos = categorias_relevantes['pedidos'].mean()

oportunidades = categorias_relevantes[
    (categorias_relevantes['ticket_medio'] > media_ticket) &
    (categorias_relevantes['pedidos'] < media_pedidos)
].sort_values('ticket_medio', ascending=False)

# Garantindo tradução das categorias no DataFrame principal para o print
categorias['product_category'] = categorias['product_category'].map(traducao_categorias).fillna(categorias['product_category'])

# Tradução para o DataFrame de oportunidades (casos específicos de nicho)
traducao_nicho = {
    'computers': 'Computadores',
    'small_appliances': 'Eletroportáteis',
    'musical_instruments': 'Instrumentos Musicais',
    'agro_industry_and_commerce': 'Agro e Indústria',
    'home_appliances_2': 'Eletrodomésticos 2'
}
oportunidades['product_category'] = oportunidades['product_category'].map(traducao_nicho).fillna(oportunidades['product_category'])

print("=" * 55)
print("     TOP 5 CATEGORIAS POR RECEITA TOTAL")
print("=" * 55)
for _, row in categorias.head(5).iterrows():
    print(f"  {row['product_category']:<30} R$ {row['receita']:>10,.2f}")

print("\n" + "=" * 55)
print("     OPORTUNIDADES — ALTO TICKET, BAIXO VOLUME")
print("=" * 55)
print(f"{'Categoria':<30} {'Ticket Médio':>12} {'Pedidos':>8}")
print("-" * 55)
for _, row in oportunidades.head(5).iterrows():
    print(f"  {row['product_category']:<28} R$ {row['ticket_medio']:>8,.2f} {row['pedidos']:>8,}")
print("=" * 55)


     TOP 5 CATEGORIAS POR RECEITA TOTAL
  Beleza e Saúde                 R$ 1,255,560.16
  Relógios e Presentes           R$ 1,197,907.21
  Cama, Mesa e Banho             R$ 1,035,964.06
  Esporte e Lazer                R$ 979,561.92
  Informática e Acessórios       R$ 904,211.03

     OPORTUNIDADES — ALTO TICKET, BAIXO VOLUME
Categoria                      Ticket Médio  Pedidos
-------------------------------------------------------
  Computadores                 R$ 1,098.34      181
  Eletrodomésticos 2           R$   470.85      231
  Agro e Indústria             R$   342.12      182
  Instrumentos Musicais        R$   280.70      620
  Eletroportáteis              R$   280.04      622


### 1.5. Análise Geográfica de Vendas
Distribuição de receita e pedidos por estado brasileiro.  
SP, RJ e MG concentram a maior parte da receita da plataforma.

In [46]:
import plotly.express as px

# Agrupamento Geográfico
geo = df.groupby('customer_state').agg(
    receita=('price', 'sum'),
    pedidos=('order_id', 'nunique'),
    ticket_medio=('price', 'mean')
).reset_index().sort_values('receita', ascending=False)

# Cálculos de Concentração Estratégica
receita_nacional = geo['receita'].sum()
geo['pct_total'] = (geo['receita'] / receita_nacional * 100).round(1)
top3_receita = geo.head(3)['receita'].sum()
pct_top3_nacional = (top3_receita / receita_nacional * 100).round(0)

# Preparando dados e cores (Revertendo para o cinza #808080)
geo_plot = geo.sort_values('receita')
cores_geo = ['#808080'] * (len(geo_plot) - 1) + ['#1f77b4']

# Rótulos Customizados com Formatação Brasileira (Vírgula)
labels_geo = []
for v, p in zip(geo_plot['receita'], geo_plot['pct_total']):
    p_str = f"{p:.1f}%".replace('.', ',')
    if v >= 1_000_000:
        valor_fmt = f"R$ {v/1_000_000:.1f}M".replace('.', ',')
    else:
        valor_fmt = f"R$ {v/1_000:.0f}K".replace('.', ',')
    labels_geo.append(f"{valor_fmt} ({p_str})")

fig = px.bar(
    geo_plot,
    x='receita',
    y='customer_state',
    orientation='h',
    title=f'<b>São Paulo Lidera Faturamento Nacional (R$ 5,2M)</b><br><span style="font-size:13px; color:rgba(255,255,255,0.6);">Dados consolidados (Out 2016 a Ago 2018) | Top 3 Estados concentram ~{pct_top3_nacional:.0f}% da receita total</span>',
    text=labels_geo
)

# Estilização das Barras e Rótulos
fig.update_traces(
    marker_color=cores_geo,
    textposition='outside',
    textfont=dict(size=11, color='rgba(255,255,255,0.8)'),
    cliponaxis=False,
    width=0.85
)

fig.update_layout(
    layout_transparente,
    title_x=0,
    height=750,
    showlegend=False,
    margin=dict(l=10, r=160, t=100, b=60),
    bargap=0.1
)

# Remoção Total do Eixo X
fig.update_xaxes(
    showgrid=False,
    showline=False,
    zeroline=False,
    showticklabels=False,
    title=None
)

fig.update_yaxes(
    ticks='outside',
    ticklen=5,
    tickcolor='rgba(0,0,0,0)',
    showgrid=False,
    showline=False,
    zeroline=False,
    title=None,
    automargin=True,
    tickfont=dict(size=12, color='white')
)

# Nota de rodapé discreta
fig.add_annotation(
    text="Fonte: Dados Olist (Out 2016 a Ago 2018)",
    xref="paper", yref="paper",
    x=1.1, y=-0.05, showarrow=False,
    font=dict(size=10, color='rgba(255,255,255,0.3)'),
    align="right"
)

fig.show()

In [47]:
top3 = geo.head(3)
pct_top3 = top3['receita'].sum() / geo['receita'].sum() * 100
pct_pedidos_top3 = top3['pedidos'].sum() / geo['pedidos'].sum() * 100

print("=" * 55)
print("        CONCENTRAÇÃO GEOGRÁFICA — TOP 3 ESTADOS")
print("=" * 55)
print(f"{'Estado':<10} {'Receita':>14} {'Pedidos':>10} {'% Receita':>10}")
print("-" * 55)
for _, row in top3.iterrows():
    pct = row['receita'] / geo['receita'].sum() * 100
    print(f"  {row['customer_state']:<8} R$ {row['receita']:>12,.2f} {row['pedidos']:>10,} {pct:>9.1f}%")
print("-" * 55)
print(f"  {'TOP 3':<8} R$ {top3['receita'].sum():>12,.2f} {top3['pedidos'].sum():>10,} {pct_top3:>9.1f}%")
print("=" * 55)
print(f"\n💡 Os 3 maiores estados concentram {pct_top3:.1f}% da receita")
print(f"   e {pct_pedidos_top3:.1f}% dos pedidos totais")

print("\n\nBottom 5 estados — menor receita:")
print(geo.tail(5)[['customer_state', 'receita', 'pedidos']].to_string(index=False))

        CONCENTRAÇÃO GEOGRÁFICA — TOP 3 ESTADOS
Estado            Receita    Pedidos  % Receita
-------------------------------------------------------
  SP       R$ 5,162,517.07     41,116      38.3%
  RJ       R$ 1,811,623.42     12,697      13.4%
  MG       R$ 1,573,508.20     11,496      11.7%
-------------------------------------------------------
  TOP 3    R$ 8,547,648.69     65,309      63.3%

💡 Os 3 maiores estados concentram 63.3% da receita
   e 66.5% dos pedidos totais


Bottom 5 estados — menor receita:
customer_state  receita  pedidos
            RO 46031.64      246
            AM 22356.84      147
            AC 15982.95       81
            AP 13474.30       68
            RR  7666.55       44


### 1.6. Satisfação dos Clientes

Distribuição das avaliações e relação entre satisfação e receita.  
A plataforma mantém avaliação média de 4.04 com 77% dos pedidos avaliados com nota 4 ou 5.

In [48]:
import plotly.express as px

# 1. Preparação dos dados
df_avaliados = df[(df['pedido_item_seq'] == 1) & (df['tem_avaliacao'] == True)].copy()
score_counts = df_avaliados['review_score'].value_counts().sort_index().reset_index()
score_counts.columns = ['score', 'quantidade']

total_avaliacoes = score_counts['quantidade'].sum()
score_counts['percentual'] = (score_counts['quantidade'] / total_avaliacoes * 100).round(1)

# Formatação de vírgula para o padrão BR
labels_br = [f"{p:.1f}%".replace('.', ',') for p in score_counts['percentual']]

# 2. Cores Estratégicas
cores_score = ['#808080', '#808080', '#808080', '#808080', '#1f77b4']

# 3. Construção do Gráfico
fig = px.bar(
    score_counts,
    x='score',
    y='quantidade',
    title='<b>Foco na Excelência: Notas Máximas Dominam Satisfação do Cliente</b><br><span style="font-size:13px; color:rgba(255,255,255,0.6);">Distribuição de satisfação consolidada (Out 2016 a Ago 2018)</span>',
    text=labels_br
)

# 4. Refinamento de Traços - Rótulos nítidos (font 13px)
fig.update_traces(
    marker_color=cores_score,
    textposition='outside',
    textfont=dict(size=13, color='rgba(255,255,255,0.9)', family='Arial'),
    cliponaxis=False,
    marker_line_width=0,
    width=0.94
)

# 5. Layout Executivo - Alinhamento à Esquerda e Margens de Respiro Lateral
fig.update_layout(
    layout_transparente,
    title_x=0.03,
    title_y=0.94,
    height=420,
    showlegend=False,
    margin=dict(t=100, b=60, l=60, r=60), # Margens laterais aumentadas para respiro visual
    bargap=0.01
)

# 6. Eixos Clean
fig.update_xaxes(
    tickmode='linear',
    showgrid=False,
    showline=True,
    linecolor='rgba(255,255,255,0.1)',
    title_text=None,
    tickfont=dict(size=14, color='rgba(255,255,255,0.9)')
)

fig.update_yaxes(
    showgrid=False,
    showline=False,
    zeroline=False,
    showticklabels=False,
    title=None
)

# Nota de Rodapé
fig.add_annotation(
    text="Fonte: Dados Olist (Out 2016 a Ago 2018)",
    xref="paper", yref="paper",
    x=1, y=-0.22, showarrow=False,
    font=dict(size=10, color='rgba(255,255,255,0.3)'),
    align="right"
)

fig.show()

In [49]:
import plotly.express as px

# 1. Preparação dos Dados
# 1. Primeiro descobrirmos o valor TOTAL de cada pedido (somando todos os seus itens)
pedidos_totais = df.groupby('order_id')['price'].sum().reset_index()

# 2. Resgatamos a nota unica do pedido usando df_avaliados
notas = df_avaliados[['order_id', 'review_score']]

# 3. Cruzamos a informacao para calcular a media
ticket_por_score = pedidos_totais.merge(notas, on='order_id').groupby('review_score').agg(
    ticket_medio=('price', 'mean')
).reset_index()

# 2. Estratégia de Cores (Destaque Laranja #e67e22 vs Cinza Uniforme)
cores_paradoxo = ['#e67e22', '#808080', '#808080', '#808080', '#808080']

# Formatação de Rótulos (Padrão Brasileiro R$ XX,XX)
labels_custom = [f"<b>R$ {v:,.2f}</b>".replace(',', 'v').replace('.', ',').replace('v', '.') for v in ticket_por_score['ticket_medio']]

# 3. Construção do Gráfico (Senior BI Standard)
fig = px.bar(
    ticket_por_score,
    x='review_score',
    y='ticket_medio',
    title='<b>Paradoxo do Valor: Itens de Maior Ticket Médio Lideram a Insatisfação</b><br><span style="font-size:13px; color:rgba(255,255,255,0.7);">Consumidores que investem mais (R$ 165) são os que atribuem as menores notas.</span>',
    text=labels_custom
)

# 4. Refinamento de Traços
fig.update_traces(
    marker_color=cores_paradoxo,
    textposition='outside',
    textfont=dict(size=13, color='rgba(255,255,255,0.9)', family='Arial'),
    cliponaxis=False,
    width=0.75
)

# 5. Layout Executivo
fig.update_layout(
    layout_transparente,
    title_x=0.01,
    height=480,
    showlegend=False,
    margin=dict(t=120, b=80, l=60, r=60),
    bargap=0.25
)

# Limpeza de Eixos
fig.update_xaxes(
    tickmode='linear',
    showgrid=False,
    showline=True,
    linecolor='rgba(255,255,255,0.1)',
    title_text=None,
    tickfont=dict(size=14, color='rgba(255,255,255,0.9)'),
    zeroline=False
)

fig.update_yaxes(
    showgrid=False,
    showline=False,
    zeroline=False,
    showticklabels=False,
    title=None
)

# Nota de Rodapé
fig.add_annotation(
    text="Fonte: Dados Olist (Out 2016 a Ago 2018)",
    xref="paper", yref="paper",
    x=1.0, y=-0.25,
    showarrow=False,
    font=dict(size=10, color='rgba(255,255,255,0.3)'),
    align="right"
)

fig.show()

In [50]:
nota_4_5 = score_counts[score_counts['score'] >= 4]['percentual'].sum()
nota_1_2 = score_counts[score_counts['score'] <= 2]['percentual'].sum()
total_avaliados = len(df_avaliados)
total_pedidos = df['order_id'].nunique()
pct_avaliados = total_avaliados / total_pedidos * 100

ticket_nota_alta = ticket_por_score[ticket_por_score['review_score'] == 5]['ticket_medio'].values[0]
ticket_nota_baixa = ticket_por_score[ticket_por_score['review_score'] == 1]['ticket_medio'].values[0]

print("=" * 55)
print("        INSIGHTS — SATISFAÇÃO DOS CLIENTES")
print("=" * 55)
print(f"⭐ Avaliação média:          {df_avaliados['review_score'].mean():.2f} / 5.0")
print(f"📋 Pedidos avaliados:        {pct_avaliados:.1f}% do total")
print(f"✅ Notas 4 e 5 (positivos):  {nota_4_5:.1f}%")
print(f"❌ Notas 1 e 2 (negativos):  {nota_1_2:.1f}%")
print(f"💰 Ticket médio nota 5:      R$ {ticket_nota_alta:.2f}")
print(f"💸 Ticket médio nota 1:      R$ {ticket_nota_baixa:.2f}")
print("=" * 55)

        INSIGHTS — SATISFAÇÃO DOS CLIENTES
⭐ Avaliação média:          4.12 / 5.0
📋 Pedidos avaliados:        99.3% do total
✅ Notas 4 e 5 (positivos):  77.9%
❌ Notas 1 e 2 (negativos):  13.8%
💰 Ticket médio nota 5:      R$ 134.64
💸 Ticket médio nota 1:      R$ 165.02


### 1.7. Logística e Satisfação: O Ponto de Ruptura da Experiência do Cliente

In [51]:
import plotly.graph_objects as go
import pandas as pd
import numpy as np

# 1. Preparação dos Dados
df_log = df[(df['pedido_item_seq'] == 1) & (df['atraso_entrega_dias'].notna())].copy()

def categorizar_atraso(dias):
    if dias <= -5: return 'Muito Antecipado (>5d)'
    if dias < 0: return 'Antecipado (1-5d)'
    if dias == 0: return 'No Prazo'
    if dias <= 3: return 'Atraso Curto (1-3d)'
    if dias <= 7: return 'Atraso Médio (4-7d)'
    return 'Atraso Crítico (>7d)'

df_log['faixa_atraso'] = df_log['atraso_entrega_dias'].apply(categorizar_atraso)

ordem_faixas = ['Muito Antecipado (>5d)', 'Antecipado (1-5d)', 'No Prazo', 'Atraso Curto (1-3d)', 'Atraso Médio (4-7d)', 'Atraso Crítico (>7d)']
satisfacao_atraso = df_log.groupby('faixa_atraso')['review_score'].mean().reindex(ordem_faixas).reset_index()

# 2. Cores Estratégicas (Semáforo)
cores_atraso = ['#1f77b4', '#808080', '#808080', '#e67e22', '#e67e22', '#e67e22']

# Formatação PT-BR: 4,00 -> 4,0 e vírgula como separador decimal
labels_barras = []
for x in satisfacao_atraso['review_score']:
    val_fmt = f"{x:.2f}".replace('.', ',')
    if val_fmt == '4,00':
        labels_barras.append('4,0')
    else:
        labels_barras.append(val_fmt)

# 3. Construção do Gráfico
fig = go.Figure()

fig.add_trace(go.Bar(
    x=satisfacao_atraso['faixa_atraso'],
    y=satisfacao_atraso['review_score'],
    text=labels_barras,
    textposition='outside',
    marker_color=cores_atraso,
    textfont=dict(size=14, color='white'),
    cliponaxis=False  # Garante que o texto fora da barra não seja cortado
))

# 4. Linha de Referência (Média Geral)
media_geral = df_log['review_score'].mean()
fig.add_shape(
    type="line", line=dict(color="rgba(255,255,255,0.3)", width=1, dash="dot"),
    x0=-0.5, x1=5.5, y0=media_geral, y1=media_geral,
    layer='below'
)

fig.add_annotation(
    x=5.2, y=media_geral + 0.15,
    text=f"Média Geral: {media_geral:.2f}".replace('.', ','),
    showarrow=False, font=dict(color="#E0E0E0", size=12)
)

# 5. Adicionando Rodapé (Fonte) - Ajustado para evitar sobreposição
fig.add_annotation(
    text="Fonte: Dados Olist (Out 2016 a Ago 2018)",
    xref="paper", yref="paper",
    x=1.0, y=-0.4,
    showarrow=False,
    font=dict(size=10, color='rgba(255,255,255,0.4)'),
    align="right"
)

# 6. Layout Executivo - Margens e Range Ajustados para Visibilidade Total
fig.update_layout(
    layout_transparente,
    title=dict(
        text="<b>O Impacto do Atraso: A Queda Drástica na Satisfação após o 3º Dia</b><br><span style='font-size:14px; color:rgba(255,255,255,0.6);'>Notas médias de review por faixa de cumprimento de SLA (Dias entre entrega real e prometida)</span>",
        x=0.0, y=0.95
    ),
    yaxis=dict(
        showgrid=False,
        zeroline=False,
        showticklabels=False,
        title=None,
        range=[0, 5.2] # Aumenta o range para o rótulo 4,28 aparecer
    ),
    xaxis=dict(
        title=None,
        showgrid=False,
        showline=False,
        zeroline=False,
        ticks='',
        showticklabels=True,
        tickfont=dict(size=12, color='rgba(255,255,255,0.8)')
    ),
    height=550,
    margin=dict(t=140, b=150, l=60, r=60),
    bargap=0
)

fig.show()

In [52]:
# --- Cálculos para a Tabela de Insights Logísticos ---

# Filtrar pedidos entregues e com nota
df_sl = df[(df['pedido_item_seq'] == 1) & (df['atraso_entrega_dias'].notna())]

total_log = len(df_sl)
pedidos_no_prazo = len(df_sl[df_sl['atraso_entrega_dias'] <= 0])
pct_no_prazo = (pedidos_no_prazo / total_log) * 100

# Impacto Real na Nota
atraso_grave = df_sl[df_sl['atraso_entrega_dias'] > 7]
nota_atraso_grave = atraso_grave['review_score'].mean()
nota_no_prazo = df_sl[df_sl['atraso_entrega_dias'] <= 0]['review_score'].mean()

# "Custo" da Satisfação (Diferença de Nota)
gap_satisfacao = nota_no_prazo - nota_atraso_grave

print("=" * 60)
print("        INSIGHTS — PERFORMANCE LOGÍSTICA E SLA")
print("=" * 60)
print(f"✅ Pedidos no Prazo (ou adiantados):  {pct_no_prazo:.1f}%")
print(f"⚠️ Pedidos com Atraso (> 0 dias):     {100 - pct_no_prazo:.1f}%")
print("-" * 60)
print(f"⭐ Nota média (Pedidos no Prazo):     {nota_no_prazo:.2f} / 5.0")
print(f"❌ Nota média (Atraso > 7 dias):      {nota_atraso_grave:.2f} / 5.0")
print(f"📉 Impacto do Atraso Crítico:        -{gap_satisfacao:.2f} pontos na nota")
print("-" * 60)
print(f"💡 INSIGHT: Cada semana de atraso extra reduz a nota em ")
print(f"   aproximadamente {gap_satisfacao:.1f} pontos, movendo o cliente da ")
print(f"   'Satisfação' para a 'Detração' (Nota 1 e 2).")
print("=" * 60)


        INSIGHTS — PERFORMANCE LOGÍSTICA E SLA
✅ Pedidos no Prazo (ou adiantados):  93.2%
⚠️ Pedidos com Atraso (> 0 dias):     6.8%
------------------------------------------------------------
⭐ Nota média (Pedidos no Prazo):     4.27 / 5.0
❌ Nota média (Atraso > 7 dias):      1.65 / 5.0
📉 Impacto do Atraso Crítico:        -2.62 pontos na nota
------------------------------------------------------------
💡 INSIGHT: Cada semana de atraso extra reduz a nota em 
   aproximadamente 2.6 pontos, movendo o cliente da 
   'Satisfação' para a 'Detração' (Nota 1 e 2).


### 1.8.  Concentração de Marketplace: A Dependência dos Top Sellers

In [53]:
import plotly.graph_objects as go
import pandas as pd

# 1. Preparação dos Dados
seller_rank = df.groupby('seller_id')['price'].sum().sort_values(ascending=False).reset_index()
seller_rank['cum_receita'] = (seller_rank['price'].cumsum() / seller_rank['price'].sum()) * 100
seller_rank['cum_vendedores'] = ((seller_rank.index + 1) / len(seller_rank)) * 100

# Ponto de 80% da receita
ponto_80 = seller_rank[seller_rank['cum_receita'] >= 80].iloc[0]
pct_vendedores_80 = ponto_80['cum_vendedores']
pct_formatada = f"{pct_vendedores_80:.1f}".replace('.', ',')

# 2. Construção do Gráfico (Design Perfeccionista)
fig = go.Figure()

# Curva de Pareto - Interatividade na linha com proteção de layout
fig.add_trace(go.Scatter(
    x=seller_rank['cum_vendedores'],
    y=seller_rank['cum_receita'],
    mode='lines',
    name='Curva',
    line=dict(color='#1f77b4', width=4),
    fill='tozeroy',
    fillcolor='rgba(31, 119, 180, 0.1)',
    hovertemplate="<b>%{x:.1f}%</b> dos vendedores → <b>%{y:.1f}%</b> da receita<extra></extra>",
    hoverlabel=dict(bgcolor='rgba(30, 30, 30, 0.9)', font_size=13, font_family='Arial')
))

# Ponto Crítico 80/20
fig.add_trace(go.Scatter(
    x=[pct_vendedores_80],
    y=[80],
    mode='markers+text',
    marker=dict(color='white', size=12, line=dict(color='#1f77b4', width=2)),
    text=[f"Top {pct_formatada}% Vendedores"],
    textposition="middle right",
    textfont=dict(color="white", size=14),
    cliponaxis=False,
    hoverinfo='skip'
))

# Linhas de Referência
fig.add_shape(type="line", x0=0, x1=pct_vendedores_80, y0=80, y1=80, line=dict(color="rgba(255,255,255,0.2)", dash="dot", width=1))
fig.add_shape(type="line", x0=pct_vendedores_80, x1=pct_vendedores_80, y0=0, y1=80, line=dict(color="rgba(255,255,255,0.2)", dash="dot", width=1))

# 3. Layout: Proteção de Títulos e Interatividade Vertical
fig.update_layout(
    layout_transparente,
    title=dict(
        text=f"<b>Risco de Concentração: {pct_formatada}% dos Vendedores Geram 80% da Receita</b><br><span style='font-size:14px; color:rgba(255,255,255,0.6);'>Curva de Pareto indica alta dependência de um pequeno grupo de Top Sellers.</span>",
        x=0.01, y=0.95
    ),
    xaxis=dict(
        showgrid=False,
        range=[0, 100],
        zeroline=False,
        tickvals=[0, 20, 40, 60, 80, 100],
        ticksuffix='%'
    ),
    yaxis=dict(
        showgrid=False,
        range=[0, 105],
        zeroline=False,
        tickvals=[20, 40, 60, 80, 100], # Evita sobreposição na origem
        ticksuffix='%'
    ),
    hovermode='x', # Garante que o hover acompanhe o cursor sem 'flutuar' para os títulos
    showlegend=False,
    height=550,
    margin=dict(t=130, b=80, l=60, r=150) # Margem superior ampliada para segurança
)

fig.add_annotation(
    text="Fonte: Dados Olist (Out 2016 a Ago 2018)",
    xref="paper", yref="paper",
    x=1.1, y=-0.15,
    showarrow=False,
    font=dict(size=10, color='rgba(255,255,255,0.4)')
)

fig.show()

In [54]:
# --- CÉLULA DE INSIGHTS: CURVA DE PARETO DE VENDEDORES ---

# Re-calculando rapidamente os valores necessários
seller_rank = df.groupby('seller_id')['price'].sum().sort_values(ascending=False).reset_index()
seller_rank['cum_receita'] = (seller_rank['price'].cumsum() / seller_rank['price'].sum()) * 100
seller_rank['cum_vendedores'] = ((seller_rank.index + 1) / len(seller_rank)) * 100

total_sellers = len(seller_rank)
ponto_80 = seller_rank[seller_rank['cum_receita'] >= 80].iloc[0]
pct_vendedores_80 = ponto_80['cum_vendedores']
num_vendedores_80 = int(ponto_80.name + 1)

print("=" * 60)
print("        INSIGHTS — CONCENTRAÇÃO DE MARKETPLACE (PARETO)")
print("=" * 60)
print(f"🏪 Total de Vendedores Ativos:     {total_sellers:,}")
print(f"🎯 Ponto de Concentração:          {pct_vendedores_80:.1f}% dos vendedores")
print(f"                                   geram 80% da receita total.")
print("-" * 60)
print(f"💎 Elite do Marketplace:           Apenas {num_vendedores_80:,} vendedores")
print(f"                                   sustentam 80% do faturamento.")
print("-" * 60)
print(f"💡 DIAGNÓSTICO: Alta dependência operacional detectada.")
print(f"   O faturamento é resiliente em volume, mas vulnerável à")
print(f"   saída de um pequeno grupo de parceiros estratégicos.")
print("=" * 60)


        INSIGHTS — CONCENTRAÇÃO DE MARKETPLACE (PARETO)
🏪 Total de Vendedores Ativos:     3,053
🎯 Ponto de Concentração:          17.7% dos vendedores
                                   geram 80% da receita total.
------------------------------------------------------------
💎 Elite do Marketplace:           Apenas 540 vendedores
                                   sustentam 80% do faturamento.
------------------------------------------------------------
💡 DIAGNÓSTICO: Alta dependência operacional detectada.
   O faturamento é resiliente em volume, mas vulnerável à
   saída de um pequeno grupo de parceiros estratégicos.


### 1.9. Dinâmica Temporal: Picos de Consumo e Comportamento de Compra

In [55]:
import plotly.express as px
import pandas as pd

# 1. Preparação dos Dados
df_temp = df[df['pedido_item_seq'] == 1].copy()
df_temp['hora'] = df_temp['order_purchase_timestamp'].dt.hour
df_temp['dia_semana'] = df_temp['order_purchase_timestamp'].dt.day_name()

dias_ordem = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
traducao_dias = {
    'Monday': 'Segunda', 'Tuesday': 'Terça', 'Wednesday': 'Quarta',
    'Thursday': 'Quinta', 'Friday': 'Sexta', 'Saturday': 'Sábado', 'Sunday': 'Domingo'
}

heatmap_data = df_temp.groupby(['dia_semana', 'hora']).size().reset_index(name='pedidos')
heatmap_data['dia_semana'] = pd.Categorical(heatmap_data['dia_semana'], categories=dias_ordem, ordered=True)
heatmap_data['dia_pt'] = heatmap_data['dia_semana'].map(traducao_dias)

pivot_heatmap = heatmap_data.pivot(index='dia_pt', columns='hora', values='pedidos').reindex(list(traducao_dias.values()))

# 2. Construção do Heatmap (Design Sênior / Dark Mode)
# Escala de cores personalizada: Cinza escuro para Azul Vibrante
custom_colorscale = [
    [0, '#1a1a1a'],
    [0.5, '#124569'],
    [1, '#1f77b4']
]

fig = px.imshow(
    pivot_heatmap,
    labels=dict(color="Pedidos"),
    x=list(range(24)),
    y=list(traducao_dias.values()),
    color_continuous_scale=custom_colorscale,
    aspect="auto"
)

# 3. Refinamento Estético (Gaps e Limpeza)
fig.update_traces(
    xgap=2,
    ygap=2
)

fig.update_layout(
    layout_transparente,
    title=dict(
        text="<b>Pico de Conversão: Dias Úteis à Tarde Concentram Maior Volume</b><br><span style='font-size:14px; color:rgba(255,255,255,0.6);'>Densidade de pedidos evidencia janelas de oportunidade para campanhas de marketing (10h às 16h)</span>",
        x=0.01, y=0.95
    ),
    xaxis=dict(
        tickmode='linear',
        dtick=2,
        title=None,
        showgrid=False,
        zeroline=False,
        ticks='' # Remove os pequenos traços verticais (ticks)
    ),
    yaxis=dict(
        title=None,
        showgrid=False,
        zeroline=False
    ),
    coloraxis_showscale=False,
    height=480,
    margin=dict(t=130, b=80, l=10, r=10)
)

# Rodapé
fig.add_annotation(
    text="Fonte: Dados Olist (Out 2016 a Ago 2018)",
    xref="paper", yref="paper",
    x=1.0, y=-0.18,
    showarrow=False,
    font=dict(size=10, color='rgba(255,255,255,0.4)'),
    align="right"
)

fig.show()

### 1.10. Geografia do Frete: O Custo Logístico como Barreira de Venda

In [56]:
import plotly.express as px

# 1. Preparação dos Dados
frete_geog = df.groupby('customer_state')['pct_frete'].mean().sort_values(ascending=False).reset_index()

# 2. Design de Cores (Laranja de Alerta #e67e22 para os 5 estados críticos, Cinza para os demais)
frete_geog_plot = frete_geog.sort_values('pct_frete')
cores_frete = ['#808080'] * (len(frete_geog_plot) - 5) + ['#e67e22'] * 5

# Rótulos Customizados com Formatação Brasileira (Vírgula)
labels_frete = [f"{p:.1f}%".replace('.', ',') for p in frete_geog_plot['pct_frete']]

# 3. Construção do Gráfico
fig = px.bar(
    frete_geog_plot,
    x='pct_frete',
    y='customer_state',
    orientation='h',
    title=f'<b>Peso do Frete por Estado: O Desafio Logístico das Regiões Norte e Nordeste</b><br><span style="font-size:13px; color:rgba(255,255,255,0.6);">Estados das regiões Norte e Nordeste pagam as maiores taxas de frete, impactando a competitividade e a margem de lucro.</span>',
    text=labels_frete
)

# Estilização das Barras e Rótulos
fig.update_traces(
    marker_color=cores_frete,
    textposition='outside',
    textfont=dict(size=11, color='rgba(255,255,255,0.8)'),
    cliponaxis=False,
    width=0.85
)

fig.update_layout(
    layout_transparente,
    title_x=0,
    height=750,
    showlegend=False,
    margin=dict(l=10, r=160, t=100, b=60),
    bargap=0.1
)

# Remoção Total do Eixo X
fig.update_xaxes(
    showgrid=False,
    showline=False,
    zeroline=False,
    showticklabels=False,
    title=None
)

fig.update_yaxes(
    ticks='outside',
    ticklen=5,
    tickcolor='rgba(0,0,0,0)',
    showgrid=False,
    showline=False,
    zeroline=False,
    title=None,
    automargin=True,
    tickfont=dict(size=12, color='white')
)

# Nota de rodapé discreta
fig.add_annotation(
    text="Fonte: Dados Olist (Out 2016 a Ago 2018)",
    xref="paper", yref="paper",
    x=1.1, y=-0.05, showarrow=False,
    font=dict(size=10, color='rgba(255,255,255,0.3)'),
    align="right"
)

fig.show()

In [57]:
# --- CÉLULA DE INSIGHTS: GEOGRAFIA DO FRETE ---

top_5_estados = frete_geog.head(5)
media_nacional_frete = frete_geog['pct_frete'].mean()
estado_mais_caro = top_5_estados.iloc[0]['customer_state']
valor_mais_caro = top_5_estados.iloc[0]['pct_frete']

# Comparação com o estado mais barato (normalmente SP)
estado_mais_barato = frete_geog.iloc[-1]['customer_state']
valor_mais_barato = frete_geog.iloc[-1]['pct_frete']

print("=" * 60)
print("        INSIGHTS — CUSTO LOGÍSTICO REGIONAL")
print("=" * 60)
print(f"🌍 MÉDIA NACIONAL DO PESO DO FRETE: {media_nacional_frete:.1f}%")
print(f"📈 ESTADO COM MAIOR ÔNUS:           {estado_mais_caro} ({valor_mais_caro:.1f}%)")
print(f"📉 ESTADO COM MENOR ÔNUS:           {estado_mais_barato} ({valor_mais_barato:.1f}%)")
print("-" * 60)
print(f"💡 IMPACTO NO CONSUMO: Em {estado_mais_caro}, o cliente paga quase")
print(f"   {valor_mais_caro/valor_mais_barato:.1f}x mais frete do que em {estado_mais_barato}.")
print("-" * 60)
print(f"🚀 ESTRATÉGIA: Sugerido parcerias com Sellers locais no")
print(f"   Norte e Nordeste (Ship-from-Store) para reduzir o ")
print(f"   'pct_frete' e aumentar a conversão nessas regiões.")
print("=" * 60)


        INSIGHTS — CUSTO LOGÍSTICO REGIONAL
🌍 MÉDIA NACIONAL DO PESO DO FRETE: 25.9%
📈 ESTADO COM MAIOR ÔNUS:           RR (32.2%)
📉 ESTADO COM MENOR ÔNUS:           SP (18.7%)
------------------------------------------------------------
💡 IMPACTO NO CONSUMO: Em RR, o cliente paga quase
   1.7x mais frete do que em SP.
------------------------------------------------------------
🚀 ESTRATÉGIA: Sugerido parcerias com Sellers locais no
   Norte e Nordeste (Ship-from-Store) para reduzir o 
   'pct_frete' e aumentar a conversão nessas regiões.


## 2.0. Diagnóstico

### 2.1. Diagnóstico de Churn de Receita

In [58]:
import plotly.graph_objects as go
import pandas as pd

# 1. Agrupamento e Cálculos
status_fin = df_diag.groupby('order_status')['price'].sum().reset_index()

cancelado = status_fin.loc[status_fin['order_status'] == 'canceled', 'price'].values[0]
indisponivel = status_fin.loc[status_fin['order_status'] == 'unavailable', 'price'].values[0]

valido_status = ['delivered', 'shipped', 'invoiced', 'processing', 'approved']
receita_real = status_fin[status_fin['order_status'].isin(valido_status)]['price'].sum()
total_potencial = receita_real + cancelado + indisponivel

# 2. Formatação PT-BR (Vírgula)
def fmt_br_final(val, is_m=True):
    if is_m:
        return f"R$ {val/1e6:.2f}".replace('.', ',') + " M"
    return f"-R$ {val/1000:.1f}".replace('.', ',') + " k"

labels_waterfall = [
    fmt_br_final(total_potencial),
    fmt_br_final(cancelado, False),
    fmt_br_final(indisponivel, False),
    fmt_br_final(receita_real)
]

# 3. Construção do Waterfall (Senior Storyteller Design)
fig = go.Figure(go.Waterfall(
    name = "Fluxo",
    orientation = "v",
    measure = ["absolute", "relative", "relative", "total"],
    x = ["Receita Bruta<br>Potencial", "Cancelados", "Falta de<br>Estoque", "Receita<br>Efetivada"],
    textposition = "outside",
    text = labels_waterfall,
    textfont = dict(size=15, color='white', family='Arial'),
    y = [total_potencial, -cancelado, -indisponivel, receita_real],
    connector = {"line":{"color":"rgba(128, 128, 128, 0.4)", "width": 1, "dash": "solid"}}, # Alterado para solid
    decreasing = {"marker":{"color":"#e67e22", "line":{"width":0}}},
    increasing = {"marker":{"color":"#1f77b4", "line":{"width":0}}},
    totals = {"marker":{"color":"#1f77b4", "line":{"width":0}}}
))

# 4. Layout C-Level e Ajuste de Margem Superior
fig.update_layout(
    layout_transparente,
    title=dict(
        text="<b>Diagnóstico Financeiro: O Vazamento de Receita</b><br><span style='font-size:14px; color:rgba(255,255,255,0.6);'>Análise de evasão por atritos operacionais (Cancelamentos e Estoque)</span>",
        x=0.01, y=0.95
    ),
    height=580,
    margin=dict(t=160, b=100, l=60, r=60),
    showlegend=False
)

fig.update_xaxes(
    showgrid=False,
    showline=False,
    zeroline=False,
    ticks="",
    tickfont=dict(size=12, color='rgba(255,255,255,0.8)')
)

fig.update_yaxes(
    showgrid=False,
    showline=False,
    zeroline=False,
    showticklabels=False,
    title=None,
    range=[0, total_potencial * 1.15]
)

# 5. Rodapé
fig.add_annotation(
    text="Fonte: Dados Olist (Out 2016 a Ago 2018)",
    xref="paper", yref="paper",
    x=1.0, y=-0.25,
    showarrow=False,
    font=dict(size=10, color='rgba(255,255,255,0.4)'),
    align="right"
)

fig.show()

In [59]:
# --- CÉLULA DE INSIGHTS: CHURN DE RECEITA ---

print("=" * 60)
print("        DIAGNÓSTICO — CHURN DE RECEITA")
print("=" * 60)
print(f"💰 Receita Potencial (Ideal):     R$ {total_potencial:,.2f}")
print(f"✅ Receita Efetivada (Real):      R$ {receita_real:,.2f}")
print("-" * 60)
print(f"❌ Perda por Cancelamento:        - R$ {cancelado:,.2f}")
print(f"❌ Perda por Falta de Estoque:    - R$ {indisponivel:,.2f}")
print("-" * 60)
print(f"💡 DIAGNÓSTICO: O Olist deixou de faturar cerca de R$ {(cancelado+indisponivel)/1000:.0f} mil")
print(f"   devido a atritos operacionais. Esse é o 'dinheiro deixado na")
print(f"   mesa' que focaremos em mitigar com otimizações de SLA.")
print("=" * 60)


        DIAGNÓSTICO — CHURN DE RECEITA
💰 Receita Potencial (Ideal):     R$ 13,589,971.26
✅ Receita Efetivada (Real):      R$ 13,492,787.80
------------------------------------------------------------
❌ Perda por Cancelamento:        - R$ 95,175.77
❌ Perda por Falta de Estoque:    - R$ 2,007.69
------------------------------------------------------------
💡 DIAGNÓSTICO: O Olist deixou de faturar cerca de R$ 97 mil
   devido a atritos operacionais. Esse é o 'dinheiro deixado na
   mesa' que focaremos em mitigar com otimizações de SLA.


### 2.2. Diagnóstico de Culpabilidade (Vendedor vs. Transportadora)

In [60]:
import plotly.express as px
import pandas as pd

# 1. Preparação (Mantendo sua lógica original)
mask_log = (df_diag['pedido_item_seq'] == 1) & \
           (df_diag['order_status'] == 'delivered') & \
           (df_diag['dias_para_postar'].notna())

df_culp = df_diag[mask_log].copy()
df_culp['dias_transporte'] = (df_culp['order_delivered_customer_date'] - df_culp['order_delivered_carrier_date']).dt.days
df_culp['status_prazo'] = df_culp['atraso_entrega_dias'].apply(lambda x: 'Atrasado' if x > 0 else 'No Prazo')

# 2. Gráfico Base (Sem filtros de escala - Honestidade Total)
fig = px.scatter(
    df_culp.sample(min(12000, len(df_culp)), random_state=42),
    x='dias_para_postar',
    y='dias_transporte',
    color='status_prazo',
    color_discrete_map={'Atrasado': '#e67e22', 'No Prazo': '#1f77b4'},
    opacity=0.3,
    labels={'dias_para_postar': 'Dias p/ Postagem (Vendedor)', 'dias_transporte': 'Dias de Viagem (Logística)'}
)

fig.update_traces(marker=dict(size=4, line=dict(width=0)))

# 3. Destaque Visual (Retângulo que acompanha o crescimento do eixo Y)
fig.add_shape(
    type="rect", x0=0, y0=15, x1=3, y1=df_culp['dias_transporte'].max(),
    fillcolor="#e67e22", opacity=0.07, layer="below", line_width=0,
)

# 4. Linhas de SLA
fig.add_vline(x=3, line_dash="dash", line_color="rgba(255,255,255,0.3)")
fig.add_hline(y=15, line_dash="dash", line_color="rgba(255,255,255,0.3)")

# 5. Anotações Estratégicas (Uso de 'paper' para fixar nos cantos)
estilo_base = dict(showarrow=False, bgcolor="rgba(17,17,17,0.8)", borderpad=5)

# 6. Layout Executivo Transparente
fig.update_layout(
    layout_transparente,
    title=dict(
        text="<b>Logística vs. Operação: Onde Moram os Atrasos?</b><br><span style='font-size:13px; color:rgba(255,255,255,0.5);'>A Jornada da Promessa de Entrega: Identificando os principais pontos de atrito que impactam o compromisso firmado com o consumidor.</span>",
        x=0.01, y=0.94
    ),
    height=650,
    showlegend=False,
    margin=dict(t=140, b=100, l=70, r=50),
    annotations=[
        dict(x=0.05, y=0.90, xref="paper", yref="paper", text="<b>FALHA LOGÍSTICA</b>", font=dict(color="#e67e22", size=11), **estilo_base),
        dict(x=0.05, y=0.05, xref="paper", yref="paper", text="<b>EFICIÊNCIA</b>", font=dict(color="#1f77b4", size=11), **estilo_base),
        dict(x=0.95, y=0.05, xref="paper", yref="paper", text="<b>FALHA VENDEDOR</b>", font=dict(color="rgba(255,255,255,0.4)", size=11), **estilo_base),
        dict(text="Fonte: Dados Olist (2016-2018)", xref="paper", yref="paper", x=1.0, y=-0.18, showarrow=False, font=dict(size=10, color='rgba(255,255,255,0.4)'))
    ]
)

# Limpeza e Grades Sutis
fig.update_xaxes(showgrid=True, gridcolor='rgba(255,255,255,0.03)', ticks="", zeroline=False)
fig.update_yaxes(showgrid=True, gridcolor='rgba(255,255,255,0.03)', ticks="", zeroline=False)

fig.show()

### 2.3. "Mapa do Churn": Geografia dos Cancelamentos

In [124]:
import pandas as pd
import plotly.express as px

# 1. Preparação dos Dados
df_pedidos = df_diag[df_diag['pedido_item_seq'] == 1].copy()
total_estado = df_pedidos.groupby('customer_state')['order_id'].count()
cancelados_estado = df_pedidos[df_pedidos['order_status'] == 'canceled'].groupby('customer_state')['order_id'].count()

df_churn_geo = pd.DataFrame({
    'Total': total_estado,
    'Cancelados': cancelados_estado
}).fillna(0)

df_churn_geo['Taxa_Churn'] = (df_churn_geo['Cancelados'] / df_churn_geo['Total'] * 100)
df_churn_geo = df_churn_geo.reset_index().sort_values('Taxa_Churn', ascending=True)

# Cálculo da Média Nacional para o Benchmark
media_nacional_churn = df_churn_geo['Taxa_Churn'].mean()

# 2. Lógica Semântica de Cores ORIGINAL
def colorir_senior(x):
    if x == 0: return '#1f77b4'
    if x > 1.0: return '#e67e22'
    return '#808080'

cores_churn = [colorir_senior(x) for x in df_churn_geo['Taxa_Churn']]

# Formatação de Rótulos PT-BR
labels_churn = [f"{p:.1f}%".replace('.', ',') for p in df_churn_geo['Taxa_Churn']]

# 3. Gerar o Gráfico de Barras Horizontais
fig = px.bar(
    df_churn_geo,
    x='Taxa_Churn',
    y='customer_state',
    orientation='h',
    title="<b>Geografia dos Cancelamentos: Onde o Churn Operacional é Crítico?</b><br><span style='font-size:13px; color:rgba(255,255,255,0.6);'>Estados do Norte apresentam churn superior ao dobro da média nacional, chegando a quase 5x no caso de Roraima.</span>",
    text=labels_churn
)

# 4. Linha de Benchmark Sutil (Dotted e Low Opacity)
fig.add_vline(
    x=media_nacional_churn,
    line_dash="dot",
    line_color="rgba(255,255,255,0.2)",
    line_width=1
)

# 5. Anotação na Base do Gráfico (Eixo X)
fig.add_annotation(
    x=media_nacional_churn,
    y=-0.02, # Levemente abaixo da base do gráfico
    xref="x",
    yref="paper",
    text=f"Média Nacional: {media_nacional_churn:.2f}%".replace('.', ','),
    showarrow=False,
    font=dict(size=10, color="rgba(255,255,255,0.4)"),
    xanchor="left",
    xshift=4
)

# 6. Estilização de Traços
fig.update_traces(
    marker_color=cores_churn,
    textposition='outside',
    textfont=dict(size=11, color='white'),
    cliponaxis=False,
    width=0.85
)

# 7. Layout Executivo Transparente
fig.update_layout(
    **layout_transparente,
    title_x=0,
    height=750,
    showlegend=False,
    margin=dict(l=10, r=160, t=100, b=80),
    bargap=0.1
)

# Limpeza Total de Eixos
fig.update_xaxes(
    showgrid=False, showline=False, zeroline=False, showticklabels=False, title=None
)

fig.update_yaxes(
    ticks='outside', ticklen=5, tickcolor='rgba(0,0,0,0)',
    showgrid=False, showline=False, zeroline=False,
    title=None, automargin=True,
    tickfont=dict(size=12, color='white')
)

# 8. Nota de Rodapé
fig.add_annotation(
    text="Fonte: Dados Olist (Out 2016 a Ago 2018)",
    xref="paper", yref="paper",
    x=1.1, y=-0.05,
    showarrow=False,
    font=dict(size=10, color='rgba(255,255,255,0.3)'),
    align="right"
)

fig.show()

### 2.4. Pareto de Categorias Detratoras (Quem "mata" a nota?)

In [115]:
import pandas as pd
import plotly.express as px

# 1. Preparação dos Dados
# Filtramos seq=1 para contar o sentimento por pedido único
df_detratores = df[(df['pedido_item_seq'] == 1) & (df['review_score'].isin([1, 2]))].copy()

pareto_categoria = df_detratores.groupby('product_category')['order_id'].count().reset_index()
pareto_categoria.columns = ['Categoria', 'Volume_Detracao']

# 2. Tradução das Categorias (Padrão do Projeto)
traducao_categorias = {
    'health_beauty': 'Beleza e Saúde',
    'watches_gifts': 'Relógios e Presentes',
    'bed_bath_table': 'Cama, Mesa e Banho',
    'sports_leisure': 'Esporte e Lazer',
    'computers_accessories': 'Informática e Acessórios',
    'furniture_decor': 'Móveis e Decoração',
    'housewares': 'Utilidades Domésticas',
    'auto': 'Automotivo',
    'garden_tools': 'Ferramentas de Jardim',
    'cool_stuff': 'Brinquedos e Games',
    'baby': 'Bebês',
    'telephony': 'Telefonia',
    'toys': 'Brinquedos'
}

pareto_categoria['Categoria'] = pareto_categoria['Categoria'].map(traducao_categorias).fillna(pareto_categoria['Categoria'])

# Ordenar e pegar as Top 12 para visualização completa conforme solicitado
pareto_categoria = pareto_categoria.sort_values('Volume_Detracao', ascending=True).tail(12)

# 3. Inversão Semântica de Cores (Storytelling Executivo)
# Laranja Vibrante (#e67e22) para o maior ofensor, Cinza (#808080) para as outras 11
cores_detracao = ['#808080'] * (len(pareto_categoria) - 1) + ['#e67e22']

# 4. Gerar o Gráfico de Barras Horizontais
fig = px.bar(
    pareto_categoria,
    x='Volume_Detracao',
    y='Categoria',
    orientation='h',
    title='<b>Foco na Qualidade: Onde Residem as Maiores Críticas?</b><br><span style="font-size:13px; color:rgba(255,255,255,0.7);">Ranking de Detratores: Cama, Mesa e Banho lidera o volume de avaliações 1 e 2 estrelas, sinalizando a necessidade de auditoria de produto.</span>',
    text='Volume_Detracao'
)

# 5. Estilização de Traços e Rótulos
fig.update_traces(
    marker_color=cores_detracao,
    textposition='outside',
    textfont=dict(size=12, color='white'),
    cliponaxis=False,
    width=0.85
)

# 6. Layout Executivo Transparente
fig.update_layout(
    layout_transparente,
    title_x=0,
    height=600,
    showlegend=False,
    margin=dict(l=10, r=100, t=100, b=60),
    bargap=0.1
)

# Limpeza Total de Eixos
fig.update_xaxes(
    showgrid=False, showline=False, zeroline=False, showticklabels=False, title=None
)

fig.update_yaxes(
    ticks='outside', ticklen=5, tickcolor='rgba(0,0,0,0)',
    showgrid=False, showline=False, zeroline=False,
    title=None, automargin=True,
    tickfont=dict(size=12, color='white')
)

# Nota de Rodapé
fig.add_annotation(
    text="Fonte: Dados Olist (Out 2016 a Ago 2018)",
    xref="paper", yref="paper",
    x=1.1, y=-0.1,
    showarrow=False,
    font=dict(size=10, color='rgba(255,255,255,0.3)'),
    align="right"
)

fig.show()

## 3.0. Análise Exploratória — Parte 2

### 3.1. Métodos de Pagamento

In [61]:
import pandas as pd
import plotly.express as px

# --- 0. DEFINIÇÃO DE ESTILO (Necessário para não dar erro) ---
layout_transparente = dict(
    paper_bgcolor='rgba(0,0,0,0)',
    plot_bgcolor='rgba(0,0,0,0)',
    font=dict(color='white')
)

# --- 1. CARREGAR E CRIAR O 'df_pagamentos' (O QUE ESTAVA FALTANDO) ---
# Certifique-se de que o arquivo olist_receita.parquet está no seu Colab
df = pd.read_parquet("olist_receita.parquet")

# Criamos o df_pagamentos contando 1 vez por pedido (pedido_item_seq == 1)
df_pagamentos = df[df['pedido_item_seq'] == 1]['payment_type'].value_counts().reset_index()
df_pagamentos.columns = ['Metodo', 'Pedidos']

# Tradução para os nomes ficarem bonitos no gráfico
traducao = {
    'credit_card': 'Cartão de Crédito',
    'boleto': 'Boleto',
    'voucher': 'Cupom/Voucher',
    'debit_card': 'Cartão de Débito'
}
df_pagamentos['Metodo'] = df_pagamentos['Metodo'].map(traducao)

# --- 2. O SEU CÓDIGO DE GRÁFICO (PROPORCIONANDO A PREPARAÇÃO) ---
df_pag_plot = df_pagamentos.sort_values('Pedidos', ascending=True)
total_pag = df_pag_plot['Pedidos'].sum()
df_pag_plot['pct'] = (df_pag_plot['Pedidos'] / total_pag * 100).round(1)

labels_custom = []
for pedidos, pct in zip(df_pag_plot['Pedidos'], df_pag_plot['pct']):
    p_str = f"{pct:.1f}%".replace('.', ',')
    v_str = f"{pedidos/1000:.1f}k".replace('.', ',')
    labels_custom.append(f"{v_str} ({p_str})")

cores_pag = ['#808080'] * (len(df_pag_plot) - 1) + ['#1f77b4']

fig = px.bar(
    df_pag_plot,
    x='Pedidos',
    y='Metodo',
    orientation='h',
    title='<b>Cartão de Crédito é o Método Preferencial de Compra</b><br><span style="font-size:13px; color:rgba(255,255,255,0.6);">Dominância Estratégica: O crédito processa 3 em cada 4 pedidos realizados na plataforma.</span>',
    text=labels_custom
)

fig.update_traces(
    marker_color=cores_pag,
    textposition='outside',
    textfont=dict(size=12, color='white'),
    cliponaxis=False,
    width=0.85
)

fig.update_layout(
    **layout_transparente,
    title_x=0,
    height=450,
    showlegend=False,
    margin=dict(l=0, r=180, t=100, b=60), # Aumentei um pouco a margem R para caber o texto
    bargap=0.1
)

fig.update_xaxes(showgrid=False, showline=False, zeroline=False, showticklabels=False, title=None)
fig.update_yaxes(
    ticks='outside', ticklen=2, tickcolor='rgba(0,0,0,0)',
    showgrid=False, showline=False, zeroline=False, title=None,
    automargin=True, tickfont=dict(size=12, color='white')
)

fig.add_annotation(
    text="Fonte: Dados Olist (Out 2016 a Ago 2018)",
    xref="paper", yref="paper",
    x=1.1, y=-0.12,
    showarrow=False,
    font=dict(size=10, color='rgba(255,255,255,0.3)'),
    align="right"
)

fig.show()


### 3.2. Recorrência de Clientes (Fidelidade)

In [62]:
import pandas as pd
import plotly.express as px

# 1. Processar a Recorrência (Lógica de CRM)
df_pedidos = df[df['pedido_item_seq'] == 1]
compras_por_cliente = df_pedidos.groupby('customer_unique_id')['order_id'].nunique().reset_index()
compras_por_cliente.columns = ['customer_unique_id', 'total_compras']

# Classificar entre Compra Única (New) vs Recorrente (Loyal)
def classificar_fidelidade(qtd):
    return "Recorrente (2+ compras)" if qtd > 1 else "Compra Única (1 compra)"

compras_por_cliente['Perfil_Fidelidade'] = compras_por_cliente['total_compras'].apply(classificar_fidelidade)
resumo_fidelidade = compras_por_cliente['Perfil_Fidelidade'].value_counts().reset_index()
resumo_fidelidade.columns = ['Perfil', 'Qtd_Clientes']

taxa_retencao = (len(compras_por_cliente[compras_por_cliente['total_compras'] > 1]) / len(compras_por_cliente)) * 100

# 2. Gerar o Gráfico de Barras Estilo Sênior
fig = px.bar(
    resumo_fidelidade,
    x='Perfil',
    y='Qtd_Clientes',
    color='Perfil',
    color_discrete_map={
        "Compra Única (1 compra)": "#808080",
        "Recorrente (2+ compras)": "#e67e22"
    },
    text_auto='.2s',
    title="<b>Análise de Retenção: O Desafio da Recompra na Olist</b><br><span style='font-size:13px; color:rgba(255,255,255,0.6);'>Dependência de Novos Clientes: Apenas 3 em cada 100 compradores retornam à plataforma.</span>"
)

# 3. Estilo Executivo Transparente
fig.update_traces(
    textposition='outside',
    textfont=dict(size=13, color='white'),
    cliponaxis=False,
    width=0.6
)

# 4. Placa de Destaque (Big Number) - Reposicionamento do conjunto completo
fig.add_annotation(
    x="Recorrente (2+ compras)",
    y=0.18, # Baixando a posição de destino da flecha no eixo y (referência paper)
    xref="x", yref="paper",
    text=f"Taxa de Retenção:<br><b style='font-size:22px'>{taxa_retencao:.2f}%</b>",
    showarrow=True,
    arrowhead=2,
    arrowcolor="white",
    standoff=5, # Espaço mínimo para não encostar na barra
    ax=0, ay=-50, # Mantendo o tamanho da flecha (haste de 50px)
    font=dict(size=14, color="#E67E22"),
    bgcolor="rgba(17,17,17,0.95)",
    bordercolor="rgba(255,255,255,0.15)",
    borderwidth=1,
    borderpad=12
)

fig.update_layout(
    **layout_transparente,
    xaxis_title=None,
    yaxis_title=None,
    showlegend=False,
    height=550,
    margin=dict(t=130, b=80, l=60, r=60),
    bargap=0.1
)

# Limpeza de Eixos
fig.update_xaxes(showgrid=False, showline=True, linecolor='rgba(255,255,255,0.1)')
fig.update_yaxes(showgrid=False, showline=False, zeroline=False, showticklabels=False)

# Nota de Rodapé
fig.add_annotation(
    text="Fonte: Dados Olist (Out 2016 a Ago 2018)",
    xref="paper", yref="paper",
    x=1.0, y=-0.2,
    showarrow=False,
    font=dict(size=9, color='rgba(255,255,255,0.4)'),
    align="right"
)

fig.show()

### 3.3. Peso vs. Frete

In [72]:
import pandas as pd
import plotly.express as px

# 1. Preparação dos Dados
# Garantimos o uso do df_diag já carregado para consistência
df_fisica = df_diag.dropna(subset=['product_weight_g', 'freight_value']).copy()
df_fisica = df_fisica[df_fisica['freight_value'] > 0]

# Conversão de Gramas para Quilos (kg) para um dashboard mais sênior
df_fisica['product_weight_kg'] = df_fisica['product_weight_g'] / 1000

# Sample estratégico para fluidez
df_sample = df_fisica.sample(min(15000, len(df_fisica)), random_state=42)

# 2. Construção do Gráfico com Storytelling
fig = px.scatter(
    df_sample,
    x='product_weight_kg',
    y='freight_value',
    opacity=0.25,
    color_discrete_sequence=['#e67e22'], # Laranja para representar custo/alerta
    trendline="ols",
    trendline_color_override="white",
    title="<b>Logística Física: O Peso como Driver de Custo de Frete</b><br><span style='font-size:13px; color:rgba(255,255,255,0.6);'>Análise de correlação: A inclinação da reta confirma que o peso é o principal fator de variação no frete.</span>",
    labels={'product_weight_kg': 'Peso (kg)', 'freight_value': 'Frete (R$)'}
)

# 3. Refinamento de Traços
fig.update_traces(marker=dict(size=4))

# 4. Layout Executivo Transparente
fig.update_layout(
    **layout_transparente,
    height=550,
    margin=dict(t=120, b=100, l=80, r=80),
    xaxis=dict(
        showgrid=False,
        showline=True,
        linecolor='rgba(255,255,255,0.1)',
        zeroline=False,
        title_font=dict(size=12, color='rgba(255,255,255,0.7)'),
        ticksuffix=' kg' # Adiciona a unidade diretamente no eixo
    ),
    yaxis=dict(
        showgrid=True,
        gridcolor='rgba(255,255,255,0.05)',
        zeroline=False,
        title_font=dict(size=12, color='rgba(255,255,255,0.7)')
    )
)

# 5. Nota de Rodapé
fig.add_annotation(
    text="Fonte: Dados Olist (Out 2016 a Ago 2018)",
    xref="paper", yref="paper",
    x=1.0, y=-0.22,
    showarrow=False,
    font=dict(size=10, color='rgba(255,255,255,0.3)'),
    align="right"
)

fig.show()